# Assembling the master modeling dataset 

To-do: 
- Merge EJI, age, county shapefiles, and distances together into one big GDF
- Incorporate median household income and vehicle access from Sara's data 

In [21]:
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Polygon, Point
import numpy as np
from functools import reduce

In [45]:
# setting wd 
import os
os.chdir('/users/bkung/fooddesertproject')

In [46]:
# reading in EJI data file from CSV 
EJI_data = pd.read_csv('modeling_data/EJI_2024_United_States.csv')

In [47]:
# filtering data to relevant counties: Bexar, Dallas, Tarrant, Travis, Harris
EJI_texas = EJI_data[EJI_data['STATEFP'] == 48]
EJI_relevant_counties = EJI_texas[EJI_texas['COUNTYFP'].isin([29, 453, 201, 113, 439])] # Bexar, Travis, Harris, Dallas, Tarrant

In [48]:
# filtering to relevant columns 
modeling_EJI_data = EJI_relevant_counties[
    ['COUNTY', 'GEOID', 'TRACTCE', 'E_WLKIND', 'EPL_WLKIND', 'E_TOTPOP', 
     'M_TOTPOP', 'E_POV200', 'E_NOHSDP', 'E_UNINSUR', 'E_CHD', 
     'E_DIABETES', 'E_AFAM','E_ASIAN', 'E_HISP']
    ]

In [49]:
# RETRIEVING ACS MEDIAN AGE DATA FROM API

# 1. Setup exact parameters
API_KEY = "3260419d0dd1c45a470edaf688621f89b71fa441"
STATE_FIPS = "48"    # Texas
COUNTY_LIST = ["029", "453", "201", "113", "439"] 

# Using the Data Profile endpoint and its matching Median Age variable
# DP05_0018E = Median Age (Total Population)
URL_BASE = "https://api.census.gov/data/2024/acs/acs5/profile"
VARIABLES = "NAME,DP05_0018E"

all_tracts_data = []

for county_fips in COUNTY_LIST:
    # Build URL carefully
    url = f"{URL_BASE}?get={VARIABLES}&for=tract:*&in=state:{STATE_FIPS}+county:{county_fips}&key={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # SAFETY CHECK: Verify the API returned JSON text before decoding it
        content_type = response.headers.get('Content-Type', '')
        if 'application/json' in content_type:
            json_data = response.json()
            
            headers = json_data[0]
            rows = json_data[1:]
            county_df = pd.DataFrame(rows, columns=headers)
            all_tracts_data.append(county_df)
        else:
            # If the Census Bureau sent a plain text error, print it out clearly!
            print(f"\n[Census API Error] Server returned text instead of data for county {county_fips}:")
            print(response.text.strip())

    except requests.exceptions.RequestException as e:
        print(f"Network or HTTP error for county {county_fips}: {e}")

# 3. Combine results safely
if all_tracts_data:
    final_df = pd.concat(all_tracts_data, ignore_index=True)
    
    # Convert and clean columns
    final_df["DP05_0018E"] = pd.to_numeric(final_df["DP05_0018E"], errors='coerce')
    final_df = final_df.rename(columns={"DP05_0018E": "median_age"})
    
    print("\n--- SUCCESS! First 5 rows: ---")
    print(final_df[["NAME", "tract", "median_age"]].head())
else:
    print("\nNo data was collected. Read the API text errors above to see why.")



--- SUCCESS! First 5 rows: ---
                                     NAME   tract  median_age
0  Census Tract 1101; Bexar County; Texas  110100        34.6
1  Census Tract 1103; Bexar County; Texas  110300        37.6
2  Census Tract 1105; Bexar County; Texas  110500        25.4
3  Census Tract 1106; Bexar County; Texas  110600        37.2
4  Census Tract 1107; Bexar County; Texas  110700        47.3


In [50]:
# reecoding missing values 
final_df['median_age'] = final_df['median_age'].replace({-666666666.0: np.nan})

In [51]:
# converting 'tract' to int 
final_df['tract'] = final_df['tract'].astype('Int64')

In [52]:
# renaming columns 
modeling_EJI_data = modeling_EJI_data.rename(
    columns={'TRACTCE': 'tract', 'E_WLKIND': 'walking_ind', 'EPL_WLKIND':'walkind_inv_perc', 'E_TOTPOP': 'total_pop', 'M_TOTPOP': 'total_pop_moe', 
             'E_POV200': 'below_200_fed_poverty_percentage', 'E_NOHSDP':'no_hs_diploma', 'E_UNINSUR': 'uninsured'})


In [53]:
# merging median age and EJI data 
EJI_age = pd.merge(modeling_EJI_data, final_df, how='left', on='tract')
EJI_age = EJI_age.drop(labels=['NAME', 'state', 'county'], axis=1)

In [54]:
EJI_age.head()

,COUNTY,GEOID,tract,walking_ind,walkind_inv_perc,total_pop,total_pop_moe,below_200_fed_poverty_percentage,no_hs_diploma,uninsured,E_CHD,E_DIABETES,E_AFAM,E_ASIAN,E_HISP,median_age
0,Tarrant County,48439102700,102700,14.6250,0.1191,3326,464,14.0709,10.1,14.1,5.8,10.3,0.9,4.5,20.7,43.1
1,Tarrant County,48439104503,104503,15.0000,0.0944,2537,407,61.9629,63.8,39.3,8.0,19.8,0.0,4.1,94.6,34.2
2,Tarrant County,48439104804,104804,13.1667,0.2323,2774,807,56.3314,34.4,18.9,6.1,16.2,0.0,3.4,84.2,27.5
3,Tarrant County,48439105600,105600,11.9167,0.3173,5236,535,25.6302,15.2,14.6,5.8,12.3,5.1,0.0,57.2,29.6
4,Tarrant County,48439106101,106101,10.7500,0.3830,1892,356,59.8464,30.2,28.7,10.0,23.6,43.1,6.3,39.1,36.0


In [55]:
# splitting them up to facilitate merges with SNAP data 
bexar_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Bexar County']
travis_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Travis County']
harris_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Harris County']
dallas_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Dallas County']
tarrant_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Tarrant County']

In [56]:
# reading in SNAP data
bexar_snap = pd.read_csv('Bexar_Data/cleaned_bexar_snap_data.csv')
travis_snap = pd.read_csv('Travis_Data/cleaned_travis_snap_data.csv')
harris_snap = pd.read_csv('Harris_Data/cleaned_harris_snap_data.csv')
dallas_snap = pd.read_csv('Dallas_Data/cleaned_dallas_snap_data.csv')
tarrant_snap = pd.read_csv('Tarrant_Data/cleaned_tarrant_snap_data.csv')

In [57]:
# separating farmers markets
bexar_fm = bexar_snap[bexar_snap['Store_Type'] == 'Farmers and Markets']
bexar_grocery= bexar_snap[bexar_snap['Store_Type'] != 'Farmers and Markets']
travis_fm = travis_snap[travis_snap['Store_Type'] == 'Farmers and Markets']
travis_grocery = travis_snap[travis_snap['Store_Type'] != 'Farmers and Markets']
harris_fm = harris_snap[harris_snap['Store_Type'] == 'Farmers and Markets']
harris_grocery = harris_snap[harris_snap['Store_Type'] != 'Farmers and Markets']
dallas_fm = dallas_snap[dallas_snap['Store_Type'] == 'Farmers and Markets']
dallas_grocery = dallas_snap[dallas_snap['Store_Type'] != 'Farmers and Markets']
tarrant_fm = tarrant_snap[tarrant_snap['Store_Type'] == 'Farmers and Markets']
tarrant_grocery = tarrant_snap[tarrant_snap['Store_Type'] != 'Farmers and Markets']

In [58]:
# converting to gdfs 
bexar_fm_gdf = gpd.GeoDataFrame(bexar_fm, geometry=gpd.points_from_xy(bexar_fm.Longitude, bexar_fm.Latitude), crs='EPSG:4326')
bexar_grocery_gdf = gpd.GeoDataFrame(bexar_grocery, geometry=gpd.points_from_xy(bexar_grocery.Longitude, bexar_grocery.Latitude), crs='EPSG:4326')

travis_fm_gdf = gpd.GeoDataFrame(travis_fm, geometry=gpd.points_from_xy(travis_fm.Longitude, travis_fm.Latitude), crs='EPSG:4326')
travis_grocery_gdf = gpd.GeoDataFrame(travis_grocery, geometry=gpd.points_from_xy(travis_grocery.Longitude, travis_grocery.Latitude), crs='EPSG:4326')

harris_fm_gdf = gpd.GeoDataFrame(harris_fm, geometry=gpd.points_from_xy(harris_fm.Longitude, harris_fm.Latitude), crs='EPSG:4326')
harris_grocery_gdf = gpd.GeoDataFrame(harris_grocery, geometry=gpd.points_from_xy(harris_grocery.Longitude, harris_grocery.Latitude), crs='EPSG:4326')

dallas_fm_gdf = gpd.GeoDataFrame(dallas_fm, geometry=gpd.points_from_xy(dallas_fm.Longitude, dallas_fm.Latitude), crs='EPSG:4326')
dallas_grocery_gdf = gpd.GeoDataFrame(dallas_grocery, geometry=gpd.points_from_xy(dallas_grocery.Longitude, dallas_grocery.Latitude), crs='EPSG:4326')

tarrant_fm_gdf = gpd.GeoDataFrame(tarrant_fm, geometry=gpd.points_from_xy(tarrant_fm.Longitude, tarrant_fm.Latitude), crs='EPSG:4326')
tarrant_grocery_gdf = gpd.GeoDataFrame(tarrant_grocery, geometry=gpd.points_from_xy(tarrant_grocery.Longitude, tarrant_grocery.Latitude), crs='EPSG:4326')

In [59]:
tarrant_grocery_gdf['Store_Type'].value_counts()

Store_Type
Super Store      130
Grocery Store     97
Supermarket       96
Other             43
Name: count, dtype: int64

In [60]:
# Reading in tract shapefiles
tracts_shp = gpd.read_file("food_justice/bexar_county/tl_2024_48_tract.shp")

# projecting to distance-friendly CRS 
bexar_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "029"]
bexar_tracts_projected = bexar_tracts.to_crs('EPSG:2278') 

travis_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "453"]
travis_tracts_projected = travis_tracts.to_crs('EPSG:2277') 

harris_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "201"]
harris_tracts_projected = harris_tracts.to_crs('EPSG:2277') 

dallas_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "113"]
dallas_tracts_projected = dallas_tracts.to_crs('EPSG:2276') 

tarrant_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "439"]
tarrant_tracts_projected = tarrant_tracts.to_crs('EPSG:2276') 

# doing the same for the farmers markets/grocery stores
bexar_fm_gdf = bexar_fm_gdf.to_crs('EPSG:2278')
bexar_grocery_gdf = bexar_grocery_gdf.to_crs('EPSG:2278')

travis_fm_gdf = travis_fm_gdf.to_crs('EPSG:2277')
travis_grocery_gdf = travis_grocery_gdf.to_crs('EPSG:2277')

harris_fm_gdf = harris_fm_gdf.to_crs('EPSG:2277')
harris_grocery_gdf = harris_grocery_gdf.to_crs('EPSG:2277')

dallas_fm_gdf = dallas_fm_gdf.to_crs('EPSG:2276')
dallas_grocery_gdf = dallas_grocery_gdf.to_crs('EPSG:2276')

tarrant_fm_gdf = tarrant_fm_gdf.to_crs('EPSG:2276')
tarrant_grocery_gdf = tarrant_grocery_gdf.to_crs('EPSG:2276')

# Computing centroids 
bexar_tracts_projected["centroid"] = bexar_tracts_projected.geometry.centroid

travis_tracts_projected["centroid"] = travis_tracts_projected.geometry.centroid

harris_tracts_projected["centroid"] = harris_tracts_projected.geometry.centroid

dallas_tracts_projected["centroid"] = dallas_tracts_projected.geometry.centroid

tarrant_tracts_projected["centroid"] = tarrant_tracts_projected.geometry.centroid

# 3. Creating new GDFs with centroids as main geometry 
bexar_centroids_projected = bexar_tracts_projected.set_geometry("centroid")
travis_centroids_projected = travis_tracts_projected.set_geometry("centroid")
harris_centroids_projected= harris_tracts_projected.set_geometry("centroid")
dallas_centroids_projected = dallas_tracts_projected.set_geometry("centroid")
tarrant_centroids_projected = tarrant_tracts_projected.set_geometry("centroid")

# 4. Find the nearest point and calculate the distance
# 'distance_col' automatically creates a column with the exact distance
bexar_nearest_fm = gpd.sjoin_nearest(
    bexar_centroids_projected, 
    bexar_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

bexar_nearest_grocery = gpd.sjoin_nearest(
    bexar_centroids_projected, 
    bexar_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

travis_nearest_fm = gpd.sjoin_nearest(
    travis_centroids_projected, 
    travis_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

travis_nearest_grocery = gpd.sjoin_nearest(
    travis_centroids_projected, 
    travis_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

harris_nearest_fm = gpd.sjoin_nearest(
    harris_centroids_projected, 
    harris_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

harris_nearest_grocery = gpd.sjoin_nearest(
    harris_centroids_projected, 
    harris_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

dallas_nearest_fm = gpd.sjoin_nearest(
    dallas_centroids_projected, 
    dallas_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

dallas_nearest_grocery = gpd.sjoin_nearest(
    dallas_centroids_projected, 
    dallas_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

tarrant_nearest_fm = gpd.sjoin_nearest(
    tarrant_centroids_projected, 
    tarrant_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

tarrant_nearest_grocery = gpd.sjoin_nearest(
    tarrant_centroids_projected, 
    tarrant_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

In [61]:
# merging farmers market and grocery stores 
bexar_distances = bexar_nearest_grocery.merge(bexar_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
bexar_distances = bexar_distances.drop(columns=[col for col in bexar_distances.columns if col.endswith("_drop")])

travis_distances = travis_nearest_grocery.merge(travis_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
travis_distances = travis_distances.drop(columns=[col for col in travis_distances.columns if col.endswith("_drop")])

harris_distances = harris_nearest_grocery.merge(harris_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
harris_distances = harris_distances.drop(columns=[col for col in harris_distances.columns if col.endswith("_drop")])

dallas_distances = dallas_nearest_grocery.merge(dallas_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
dallas_distances = dallas_distances.drop(columns=[col for col in dallas_distances.columns if col.endswith("_drop")])

tarrant_distances = tarrant_nearest_grocery.merge(tarrant_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
tarrant_distances = tarrant_distances.drop(columns=[col for col in tarrant_distances.columns if col.endswith("_drop")])

In [62]:
# cleaning up columns
bexar_distances = bexar_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
travis_distances = travis_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
harris_distances = harris_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
dallas_distances = dallas_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
tarrant_distances = tarrant_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]

In [63]:
# Filtering Travis County Farms 
farms_to_keep = [
    "Urban Roots East Austin Farm", 
    "Urban Roots South Austin Farm",
    "HausBar Urban Farm",
    "Historic Boggy Creek Farm",
    "Green Gate Farms",
    "Gray Fox Market Garden",
    "Agua Dulce Austin",
    "Aquaflora on Evelyn",
    "Farmshare Austin",
    "New Leaf Agriculture",
    "Patchwork Farm"
]

# 2. Load your uploaded CSV file
# Make sure the file name matches exactly
df = pd.read_csv('final_urbanfarm_data/final_travis_farms.csv')

# 3. Filter the data to keep only the rows where the 'name' column is in our list
travis_final = df[df['name'].isin(farms_to_keep)]

# 4. Export the filtered data to a new CSV file
# index=False ensures we don't add an extra column of row numbers
travis_final.to_csv('final_urbanfarm_data/final_travis_farms.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'final_urbanfarm_data/final_travis_farms.csv'

In [64]:
# loading in urban farms data 
bexar_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_bexar.csv')
travis_uf = pd.read_csv('final_urbanfarm_data/final_travis_farms.csv')
harris_uf = pd.read_excel('final_urbanfarm_data/harris_farms_final.xlsx')
dallas_uf = pd.read_csv('final_urbanfarm_data/final_dallas_farms.csv')
tarrant_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_tarrant.csv')

# converting to gdfs
bexar_uf_gdf = gpd.GeoDataFrame(bexar_uf, geometry=gpd.points_from_xy(bexar_uf.lon, bexar_uf.lat), crs='EPSG:4326')
travis_uf_gdf = gpd.GeoDataFrame(travis_uf, geometry=gpd.points_from_xy(travis_uf.lon, travis_uf.lat), crs='EPSG:4326')
harris_uf_gdf = gpd.GeoDataFrame(harris_uf, geometry=gpd.points_from_xy(harris_uf.lon, harris_uf.lat), crs='EPSG:4326')
dallas_uf_gdf = gpd.GeoDataFrame(dallas_uf, geometry=gpd.points_from_xy(dallas_uf.lon, dallas_uf.lat), crs='EPSG:4326')
tarrant_uf_gdf = gpd.GeoDataFrame(tarrant_uf, geometry=gpd.points_from_xy(tarrant_uf.lon, tarrant_uf.lat), crs='EPSG:4326')

FileNotFoundError: [Errno 2] No such file or directory: 'final_urbanfarm_data/final_travis_farms.csv'

In [ ]:
# dropping unnecssary columns
bexar_uf_gdf = bexar_uf_gdf[['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']]
travis_uf_gdf = travis_uf_gdf[['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']]
harris_uf_gdf = harris_uf_gdf[['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']]
dallas_uf_gdf = dallas_uf_gdf[['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']]
tarrant_uf_gdf = tarrant_uf_gdf[['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']]

In [ ]:
# calculating distances to nearest urban farm 

# projecting farms 
bexar_uf_gdf = bexar_uf_gdf.to_crs('EPSG:2278')
travis_uf_gdf = travis_uf_gdf.to_crs('EPSG:2277')
harris_uf_gdf = harris_uf_gdf.to_crs('EPSG:2277')
dallas_uf_gdf = dallas_uf_gdf.to_crs('EPSG:2276')
tarrant_uf_gdf = tarrant_uf_gdf.to_crs('EPSG:2276')

# using previously generated centroids to generate distances to nearest urban farm 
bexar_nearest_uf = gpd.sjoin_nearest(
    bexar_centroids_projected, 
    bexar_uf_gdf, 
    how="left", 
    distance_col="distance_to_nearest_uf"
)

travis_nearest_uf = gpd.sjoin_nearest(
    travis_centroids_projected, 
    travis_uf_gdf, 
    how="left", 
    distance_col="distance_to_nearest_uf"
)

harris_nearest_uf = gpd.sjoin_nearest(
    harris_centroids_projected, 
    harris_uf_gdf, 
    how="left", 
    distance_col="distance_to_nearest_uf"
)

dallas_nearest_uf = gpd.sjoin_nearest(
    dallas_centroids_projected, 
    dallas_uf_gdf, 
    how="left", 
    distance_col="distance_to_nearest_uf"
)

tarrant_nearest_uf = gpd.sjoin_nearest(
    tarrant_centroids_projected, 
    tarrant_uf_gdf, 
    how="left", 
    distance_col="distance_to_nearest_uf"
)

In [ ]:
# RETRIEVING ACS MEDIAN HOUSEHOLD INCOME AND VEHICLE ACCESS DATA FROM API

# 1. Setup exact parameters
API_KEY = "3260419d0dd1c45a470edaf688621f89b71fa441"
STATE_FIPS = "48"    # Texas
COUNTY_LIST = ["029", "453", "201", "113", "439"] 

# Using the Data Profile endpoint and its matching Median Age variable
# DP05_0018E = Median Age (Total Population)
URL_BASE = "https://api.census.gov/data/2024/acs/acs5"
VARIABLES = "NAME,B19013_001E,B08201_002E,B08201_001E" # household income, households with no vehicle access, total households 

all_tracts_data = []

for county_fips in COUNTY_LIST:
    # Build URL carefully
    url = f"{URL_BASE}?get={VARIABLES}&for=tract:*&in=state:{STATE_FIPS}+county:{county_fips}&key={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # SAFETY CHECK: Verify the API returned JSON text before decoding it
        content_type = response.headers.get('Content-Type', '')
        if 'application/json' in content_type:
            json_data = response.json()
            
            headers = json_data[0]
            rows = json_data[1:]
            county_df = pd.DataFrame(rows, columns=headers)
            all_tracts_data.append(county_df)
        else:
            # If the Census Bureau sent a plain text error, print it out clearly!
            print(f"\n[Census API Error] Server returned text instead of data for county {county_fips}:")
            print(response.text.strip())

    except requests.exceptions.RequestException as e:
        print(f"Network or HTTP error for county {county_fips}: {e}")

# 3. Combine results safely
if all_tracts_data:
    acs_income_vehicle = pd.concat(all_tracts_data, ignore_index=True)
    
    # Convert and clean columns
    acs_income_vehicle["B19013_001E"] = pd.to_numeric(acs_income_vehicle["B19013_001E"], errors='coerce')
    acs_income_vehicle["B08201_002E"] = pd.to_numeric(acs_income_vehicle["B08201_002E"], errors='coerce')
    acs_income_vehicle["B08201_001E"] = pd.to_numeric(acs_income_vehicle["B08201_001E"], errors='coerce')
    acs_income_vehicle = acs_income_vehicle.rename(columns={"B19013_001E": "median_income", "B08201_002E": "no_vehicle", "B08201_001E": "total_households"})
    
    print("\n--- SUCCESS! First 5 rows: ---")
    print(acs_income_vehicle[["NAME", "tract", "median_income", "no_vehicle", "total_households"]].head())
else:
    print("\nNo data was collected. Read the API text errors above to see why.")

In [ ]:
# recoding missing values (again) :D 
acs_income_vehicle = acs_income_vehicle.replace({-666666666.0: np.nan})

# converting 'tract' to int 
acs_income_vehicle[['tract', 'state', 'county']] = acs_income_vehicle[['tract', 'state', 'county']].astype('Int64')

In [ ]:
# creating % households with no vehicle variable
acs_income_vehicle['pct_no_vehicle'] = acs_income_vehicle['no_vehicle'] / acs_income_vehicle['total_households'] * 100 

In [ ]:
# splitting by county to facilitate merges 
bexar_income_vehicle = acs_income_vehicle[acs_income_vehicle['county'] == 29]
travis_income_vehicle = acs_income_vehicle[acs_income_vehicle['county'] == 453]
harris_income_vehicle = acs_income_vehicle[acs_income_vehicle['county'] == 201]
dallas_income_vehicle = acs_income_vehicle[acs_income_vehicle['county'] == 113]
tarrant_income_vehicle = acs_income_vehicle[acs_income_vehicle['county'] == 439]

In [44]:
bexar_distances['tract'] = bexar_distances['TRACTCE'].astype('Int64')
bexar_nearest_uf['tract'] = bexar_nearest_uf['TRACTCE'].astype('Int64')
travis_distances['tract'] = travis_distances['TRACTCE'].astype('Int64')
travis_nearest_uf['tract'] = travis_nearest_uf['TRACTCE'].astype('Int64')
harris_distances['tract'] = harris_distances['TRACTCE'].astype('Int64')
harris_nearest_uf['tract'] = harris_nearest_uf['TRACTCE'].astype('Int64')
dallas_distances['tract'] = dallas_distances['TRACTCE'].astype('Int64')
dallas_nearest_uf['tract'] = dallas_nearest_uf['TRACTCE'].astype('Int64')
tarrant_distances['tract'] = tarrant_distances['TRACTCE'].astype('Int64')
tarrant_nearest_uf['tract'] = tarrant_nearest_uf['TRACTCE'].astype('Int64')

NameError: name 'bexar_nearest_uf' is not defined

In [43]:
# merging all of the dataframes into one for modeling 
merge_key = 'tract'

# defining merge function
def smart_merge(left, right):
    # 1. Identify columns in the 'right' DataFrame that already exist in the 'left' DataFrame
    # We make sure NOT to drop the merge_key, otherwise we can't merge!
    overlapping_cols = [col for col in right.columns if col in left.columns and col != merge_key]
    
    # 2. Drop those overlapping columns from the 'right' DataFrame
    right_cleaned = right.drop(columns=overlapping_cols)
    
    # 3. Perform the merge
    return pd.merge(left, right_cleaned, on=merge_key, how='outer')

# merging all of the counties 
bexar_dfs = [bexar_EJI_age, bexar_distances, bexar_nearest_uf, bexar_income_vehicle]
bexar_merged = reduce(smart_merge, bexar_dfs)
travis_dfs = [travis_EJI_age, travis_distances, travis_nearest_uf, travis_income_vehicle]
travis_merged = reduce(smart_merge, travis_dfs)
harris_dfs = [harris_EJI_age, harris_distances, harris_nearest_uf, harris_income_vehicle]
harris_merged = reduce(smart_merge, harris_dfs)
dallas_dfs = [dallas_EJI_age, dallas_distances, dallas_nearest_uf, dallas_income_vehicle]
dallas_merged = reduce(smart_merge, dallas_dfs)
tarrant_dfs = [tarrant_EJI_age, tarrant_distances, tarrant_nearest_uf, tarrant_income_vehicle]
tarrant_merged = reduce(smart_merge, tarrant_dfs)

NameError: name 'bexar_nearest_uf' is not defined

In [41]:
# cleaning up columns 
columns_to_keep = ['COUNTY', 'GEOID', 'tract', 'walking_ind', 'walkind_inv_perc', 'total_pop',
                             'total_pop_moe', 'below_200_fed_poverty_percentage', 'no_hs_diploma', 'uninsured', 'E_CHD', 'E_DIABETES', 
                             'E_AFAM', 'E_ASIAN', 'E_HISP', 'median_age', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm', 
                             'geometry', 'distance_to_nearest_uf', 'median_income', 'pct_no_vehicle']
bexar_merged = bexar_merged[columns_to_keep]
travis_merged = travis_merged[columns_to_keep]
harris_merged = harris_merged[columns_to_keep]
dallas_merged = dallas_merged[columns_to_keep]
tarrant_merged = tarrant_merged[columns_to_keep]

# merging and writing to files 
bexar_merged_gdf = gpd.GeoDataFrame(bexar_merged, geometry='geometry')
bexar_merged_gdf.to_file("modeling_data/bexar_modeling_final.gpkg", driver="GPKG")

travis_merged_gdf = gpd.GeoDataFrame(travis_merged, geometry='geometry')
travis_merged_gdf.to_file("modeling_data/travis_modeling_final.gpkg", driver="GPKG")

harris_merged_gdf = gpd.GeoDataFrame(harris_merged, geometry='geometry')
harris_merged_gdf.to_file("modeling_data/harris_modeling_final.gpkg", driver="GPKG")

dallas_merged_gdf = gpd.GeoDataFrame(dallas_merged, geometry='geometry')
dallas_merged_gdf.to_file("modeling_data/dallas_modeling_final.gpkg", driver="GPKG")

tarrant_merged_gdf = gpd.GeoDataFrame(tarrant_merged, geometry='geometry')
tarrant_merged_gdf.to_file("modeling_data/tarrant_modeling_final.gpkg", driver="GPKG")

NameError: name 'bexar_merged' is not defined

In [66]:
# creating master dataset 
bexar_merged_gdf = gpd.read_file("modeling_data/bexar_modeling_final.gpkg").to_crs("EPSG:3857")
travis_merged_gdf = gpd.read_file("modeling_data/travis_modeling_final.gpkg").to_crs("EPSG:3857")
harris_merged_gdf = gpd.read_file("modeling_data/harris_modeling_final.gpkg").to_crs("EPSG:3857")
dallas_merged_gdf = gpd.read_file("modeling_data/dallas_modeling_final.gpkg").to_crs("EPSG:3857")
tarrant_merged_gdf = gpd.read_file("modeling_data/tarrant_modeling_final.gpkg").to_crs("EPSG:3857")
county_dfs = [bexar_merged_gdf, travis_merged_gdf, harris_merged_gdf, dallas_merged_gdf, tarrant_merged_gdf]
modeling_data_final = pd.concat(county_dfs, ignore_index=True)

In [67]:
modeling_data_final.to_file("modeling_data/modeling_data_final.gpkg", driver="GPKG")